In [ ]:

import os
import sys
import regex as re
import plotly.express as px
import pandas as pd


# If your current working directory is the notebooks directory, use this:
notebook_dir = os.getcwd()  # current working directory
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
model_path = os.path.abspath(os.path.join(notebook_dir, '..', 'model_pipeline'))


sys.path.append(parent_dir)
sys.path.append(src_path)
sys.path.append(model_path)

import glob
import pickle
from IPython.display import Markdown
from server_config import datapath, preprocessed_path, preprocessed_path_freezed, redcap_path

import numpy as np
from scipy.stats import f_oneway


In [ ]:
with open(preprocessed_path + '/passive_adherence_data.pkl', 'rb') as file:
    df_passive = pickle.load(file)
with open(preprocessed_path + '/ema_adherence_data.pkl', 'rb') as file:
    df_ema = pickle.load(file)
with open(preprocessed_path + '/monitoring_data.pkl', 'rb') as file:
    df_monitoring = pickle.load(file)
with open(preprocessed_path + '/ema_content.pkl', 'rb') as file:
    df_ema_content = pickle.load(file)

df_redcap = pd.read_csv(redcap_path + "FOR5187_DATA_2025-04-01_1402.csv")

In [ ]:
df_ema_content.columns.tolist()

In [ ]:
df_ema.columns.tolist()

In [ ]:
df_monitoring_short = df_monitoring[["customer","for_id","status", "study_version" ]]

In [ ]:
df_redcap_short = df_redcap[['for_id','ema_smartphone']]

In [ ]:
df_passive = df_passive.merge(df_monitoring_short, on="customer", how="left")

In [ ]:
df_passive_long_reduced = df_passive.loc[df_passive.study_version.isin(["Lang", "Lang (Wechsel)"])]

In [ ]:
df_passive_long_reduced = df_passive_long_reduced.loc[df_passive_long_reduced.type.isin(['Latitude', 'Steps','HeartRate','ActivityType'])]
# Drop the unused categories from the 'type' column
df_passive_long_reduced['type'] = df_passive_long_reduced['type'].cat.remove_unused_categories()


In [ ]:
import plotly.express as px

# Filter the DataFrame for relative days between 1 and 90
df_filtered = df_passive_long_reduced[
    (df_passive_long_reduced['relative_day'] >= 1) & 
    (df_passive_long_reduced['relative_day'] <= 90)
]

# Group by "type" and "relative_day" and calculate the mean availability
availability = df_filtered.groupby(['type', 'relative_day'])['available_binary'].mean().reset_index()
availability['percentage'] = availability['available_binary'] * 100

# Create a line plot
fig = px.line(
    availability, 
    x='relative_day', 
    y='percentage', 
    color='type',
    labels={
        'relative_day': 'Study Day',
        'percentage': 'Data Availability (%)',
        'type': 'Data Type'
    },
    title=''
)

# Adjust layout for size and fonts
fig.update_layout(
    width=1000,
    height=500,
    font=dict(size=16),  # Increase base font size
    xaxis_title_font=dict(size=18),
    yaxis_title_font=dict(size=18),
    legend_title_font=dict(size=16),
    legend_font=dict(size=14)
)

# Add a dotted vertical line at day 14
fig.add_shape(
    dict(
        type="line",
        x0=14,
        x1=14,
        y0=0,
        y1=1,
        xref='x',
        yref='paper',
        line=dict(
            color="black",
            width=2,
            dash="dot"
        )
    )
)

# Add an annotation for the vertical line
fig.add_annotation(
    x=14,
    y=1,
    xref="x",
    yref="paper",
    text="End of first assessment phase",
    showarrow=True,
    arrowhead=1,
    ax=20,
    ay=-20,
    font=dict(size=16)  # Increase annotation font size
)

# Update y-axis tick spacing
fig.update_yaxes(tickmode="linear", dtick=5)

fig.show()


In [ ]:
df_passive_long_reduced = df_passive_long_reduced.merge(df_redcap_short, on="for_id", how="left")

In [ ]:
df_passive_long_reduced = df_passive_long_reduced.dropna(subset= "ema_smartphone")

In [ ]:
import plotly.express as px

# Filter for type "Latitude" and relative days between 1 and 90
df_latitude = df_passive_long_reduced[
    (df_passive_long_reduced['type'] == 'Latitude') &
    (df_passive_long_reduced['relative_day'] >= 1) &
    (df_passive_long_reduced['relative_day'] <= 90)
].copy()  # .copy() to avoid SettingWithCopyWarning

# Map 0/1 to 'Android' and 'iPhone'
df_latitude['ema_smartphone'] = df_latitude['ema_smartphone'].map({0: 'Android', 1: 'iPhone'})

# Group by 'ema_smartphone' and 'relative_day', calculate the mean availability
availability_lat = df_latitude.groupby(['ema_smartphone', 'relative_day'])['available_binary'].mean().reset_index()
availability_lat['percentage'] = availability_lat['available_binary'] * 100

# Create the line plot
fig = px.line(
    availability_lat, 
    x='relative_day', 
    y='percentage', 
    color='ema_smartphone',
    labels={
        'relative_day': 'Study Day',
        'percentage': 'Data Availability (%)',
        'ema_smartphone': 'Smartphone Type'
    },
    title='',
    color_discrete_sequence=["#228B22", "#7CFC00"]
)

# Update layout for size and fonts
fig.update_layout(
    width=1000,
    height=500,
    font=dict(size=16),
    xaxis_title_font=dict(size=18),
    yaxis_title_font=dict(size=18),
    legend_title_font=dict(size=16),
    legend_font=dict(size=14)
)

# Add vertical line at day 14
fig.add_shape(
    dict(
        type="line",
        x0=14,
        x1=14,
        y0=0,
        y1=1,
        xref='x',
        yref='paper',
        line=dict(
            color="black",
            width=2,
            dash="dot"
        )
    )
)

# Add annotation
fig.add_annotation(
    x=14,
    y=1,
    xref="x",
    yref="paper",
    text="end of first active phase",
    showarrow=True,
    arrowhead=1,
    ax=20,
    ay=-20,
    font=dict(size=16)
)

# Y-axis tick spacing
fig.update_yaxes(tickmode="linear", dtick=5)

fig.show()


In [ ]:
df_ema_short = df_ema[["customer", "study_version", "status","nquest_EMA1", "nquest_EMA2", "nquest_EMA3"]]

In [ ]:
df_ema_short = df_ema_short.dropna(subset= "study_version")

In [ ]:
df_ema_short = df_ema_short.loc[df_ema_short.status.isin(['Abgeschlossen', 'Post_Erhebung_2','Post_Erhebung_1', 
                                                          'Erhebung_2_aktiv', 'Erhebung_3_aktiv', 'Dropout'])]

In [ ]:
df_ema_short= df_ema_short.drop_duplicates()

In [ ]:
import pandas as pd
import plotly.express as px
from scipy.stats import ttest_ind

# ------------------------------------------------------------------------------
# Recode study_version and map to "Long" and "Short"
# ------------------------------------------------------------------------------
df_ema_short['study_version_reduced'] = df_ema_short['study_version'].replace({
    'Lang (Wechsel)': 'Lang',
    'Kurz (Wechsel/Abbruch)': 'Kurz'
})

# Map German labels to English
df_ema_short['study_version_reduced'] = df_ema_short['study_version_reduced'].map({
    'Lang': 'Long',
    'Kurz': 'Short'
})

MAX_QUEST = 112

# Calculate adherence percentage for EMA1
df_ema_short['percentage_EMA1'] = df_ema_short['nquest_EMA1'] / MAX_QUEST * 100
df_filtered = df_ema_short[df_ema_short['percentage_EMA1'] <= 110]

# ------------------------------------------------------------------------------
# Create boxplot
# ------------------------------------------------------------------------------
fig2 = px.box(
    df_filtered,
    x="study_version_reduced",
    y="percentage_EMA1",
    points="all",
    title="Adherence (EMA1) by Study Version (Boxplot)",
    labels={"study_version_reduced": "Study Version", "percentage_EMA1": "Adherence EMA1 (%)"}
)

# Update layout for size and fonts
fig2.update_layout(
    height=800,
    width=600,
    boxmode='group',
    boxgap=0.8,
    font=dict(size=18),  # Set global font size
    title_font=dict(size=22),
    xaxis_title_font=dict(size=20),
    yaxis_title_font=dict(size=20),
    legend_title_font=dict(size=18),
    legend_font=dict(size=16)
)

fig2.update_xaxes(tickfont=dict(size=18))
fig2.update_yaxes(tickfont=dict(size=18))

# ------------------------------------------------------------------------------
# Add mean ± SD annotations for each group
# ------------------------------------------------------------------------------
ema1_stats = df_filtered.groupby('study_version_reduced')['percentage_EMA1'] \
                          .agg(['mean', 'std']) \
                          .reset_index() \
                          .rename(columns={'mean': 'mean_percentage', 'std': 'std_percentage'})

for _, row in ema1_stats.iterrows():
    annotation_text = f"{row['mean_percentage']:.1f} ± {row['std_percentage']:.1f}"
    fig2.add_annotation(
        x=row['study_version_reduced'],
        y=row['mean_percentage'] + 2,  # slight vertical offset
        xshift=20,
        text=annotation_text,
        showarrow=False,
        font=dict(size=18),
        xanchor="left"
    )

# ------------------------------------------------------------------------------
# Perform Welch's t-test and annotate p-value
# ------------------------------------------------------------------------------
group_long = df_filtered[df_filtered['study_version_reduced'] == 'Long']['percentage_EMA1'].dropna()
group_short = df_filtered[df_filtered['study_version_reduced'] == 'Short']['percentage_EMA1'].dropna()
t_stat, p_val = ttest_ind(group_long, group_short, equal_var=False)

# Update plot title
fig2.update_layout(
    title_text=f"Adherence (EMA1) by Study Version (Boxplot) (p = {p_val:.3f})"
)

# Add p-value annotation
fig2.add_annotation(
    x=0.5, y=1.05,
    xref="paper", yref="paper",
    text=f"t-test: p = {p_val:.3f}",
    showarrow=False,
    font=dict(size=20, color="black")
)

fig2.show()


In [ ]:
import numpy as np
from scipy.stats import f_oneway
import plotly.express as px

df_ema_short = df_ema_short.loc[df_ema_short['study_version_reduced'].isin(["Long"])]

MAX_QUEST = 112

# Melt to long format
df_long = df_ema_short.melt(
    id_vars=['customer', 'study_version_reduced', 'study_version', 'status'],
    value_vars=['nquest_EMA1', 'nquest_EMA2', 'nquest_EMA3'],
    var_name='assessment_phase',
    value_name='completed_count'
)

# Calculate adherence percentage
df_long['percentage'] = df_long['completed_count'] / MAX_QUEST * 100
df_long = df_long[df_long['percentage'] <= 110]

# Compute stats per phase
phase_stats = df_long.groupby('assessment_phase').agg(
    mean_percentage=('percentage', 'mean'),
    std_percentage=('percentage', 'std'),
    count=('percentage', 'count')
).reset_index()
phase_stats['se_percentage'] = phase_stats['std_percentage'] / np.sqrt(phase_stats['count'])

# ANOVA
group_EMA1 = df_long[df_long['assessment_phase'] == 'nquest_EMA1']['percentage'].dropna()
group_EMA2 = df_long[df_long['assessment_phase'] == 'nquest_EMA2']['percentage'].dropna()
group_EMA3 = df_long[df_long['assessment_phase'] == 'nquest_EMA3']['percentage'].dropna()
f_stat, p_val_anova = f_oneway(group_EMA1, group_EMA2, group_EMA3)

# Plot
fig1 = px.line(
    phase_stats,
    x='assessment_phase',
    y='mean_percentage',
    markers=True,
    error_y='se_percentage',
    labels={
        'assessment_phase': 'Assessment Phase',
        'mean_percentage': 'Adherence (%)'
    }
)

# Customize x-axis tick labels
fig1.update_xaxes(
    tickmode="array",
    tickvals=["nquest_EMA1", "nquest_EMA2", "nquest_EMA3"],
    ticktext=["1", "2", "3"],
    tickfont=dict(size=20),
    title_font=dict(size=22)
)

# Customize y-axis
fig1.update_yaxes(
    tickfont=dict(size=20),
    title_font=dict(size=22)
)

# Layout updates
fig1.update_layout(
    height=600,
    width=800,
    title_text=f"Average Adherence Across Assessment Phases (with SE Error Bars)<br>ANOVA p = {p_val_anova:.3f}",
    title_font=dict(size=24),
    font=dict(size=18)
)

# Line color
fig1.update_traces(line=dict(color='green'))

# Annotate each point with mean ± SD
for _, row in phase_stats.iterrows():
    annotation_text = f"{row['mean_percentage']:.1f} ± {row['std_percentage']:.1f}"
    fig1.add_annotation(
        x=row['assessment_phase'],
        y=row['mean_percentage'] + row['se_percentage'] + 2,
        text=annotation_text,
        showarrow=False,
        font=dict(size=18)
    )

# Centered ANOVA annotation
fig1.add_annotation(
    x=0.5,
    y=1.10,
    xref="paper",
    yref="paper",
    text=f"One-way ANOVA p = {p_val_anova:.3f}",
    showarrow=False,
    font=dict(size=20, color="black")
)

fig1.show()
